<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); padding: 40px; border-radius: 10px; margin-bottom: 20px;">
<h1 style="color: white; margin: 0; font-size: 2.5em;">Lecture 6: Memory & Storage</h1>
<p style="color: #888; font-size: 1.2em; margin-top: 10px;">Part II: Devices & Circuits - SCE Futures</p>
</div>

## Contents

- [Learning Objectives](#learning-objectives)

1. [The Memory Challenge in SCE](#1-memory-challenge)
2. [Shift Register Memory](#2-shift-registers)
3. [Vortex Transitional Memory](#3-vortex-memory)
4. [SFQ-Based Storage Cells](#4-sfq-storage)
5. [Register Files & FIFOs](#5-register-fifos)
6. [External Memory Integration](#6-external-memory)
7. [Memory for AI Workloads](#7-memory-ai)

- [Summary](#summary)

---
<a id="learning-objectives"></a>
## Learning Objectives

By the end of this session, you will be able to:

- Explain why traditional SRAM architecture doesn't translate to superconducting electronics
- Describe shift register and vortex-based memory approaches
- Analyze trade-offs between different SCE storage architectures
- Design register files and FIFOs using SCE primitives
- Understand interface requirements for external memory (HBM)
- Match memory architecture to workload requirements

In [ ]:
# Setup
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np

COLORS = {
    'primary': '#2196F3',
    'secondary': '#FF9800',
    'success': '#4CAF50',
    'danger': '#f44336',
    'dark': '#1a1a2e',
    'light': '#f5f5f5',
    'purple': '#9C27B0',
    'cyan': '#00BCD4',
    'sram': '#E91E63',
    'fifo': '#4CAF50',
    'register': '#FF9800',
}

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 11

print("Setup complete.")

---
<a id="1-memory-challenge"></a>
# 1. The Memory Challenge in SCE
---

## Why CMOS SRAM Doesn't Translate

In CMOS, a 6-transistor (6T) SRAM cell stores a bit using cross-coupled inverters:

- Two stable states (0 or 1) maintained by transistor feedback
- Static power near zero (only leakage)
- Random access: any cell addressable in O(1) time

**The problem for SCE:**

| CMOS SRAM Property | SCE Reality |
|-------------------|-------------|
| Transistor bistability | No transistors in JJ logic |
| Static storage | JJ circuits are inherently dynamic (SFQ pulses) |
| Dense 6T cells | JJ cells much larger (inductors, JJs) |
| Room-temp compatible | 4K operation requires cryo interface |

### The Fundamental Issue

Josephson junctions don't have stable "on" and "off" states like transistors. Information in SFQ logic is encoded in the **presence or absence of flux quanta** (voltage pulses), which are transient events.

To store a bit, we need to:
1. Capture a flux quantum in a superconducting loop, or
2. Continuously circulate data through a shift register

In [ ]:
# Visualize: CMOS SRAM vs SCE storage concepts
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# CMOS 6T SRAM
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('CMOS 6T SRAM Cell', fontsize=12, fontweight='bold')

# Draw cross-coupled inverters
inv1 = FancyBboxPatch((2, 4), 2, 2, boxstyle="round,pad=0.1", 
                       facecolor=COLORS['sram'], edgecolor='black', linewidth=2)
inv2 = FancyBboxPatch((6, 4), 2, 2, boxstyle="round,pad=0.1",
                       facecolor=COLORS['sram'], edgecolor='black', linewidth=2)
ax.add_patch(inv1)
ax.add_patch(inv2)
ax.text(3, 5, 'INV', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(7, 5, 'INV', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Cross-coupling arrows
ax.annotate('', xy=(6, 5.5), xytext=(4, 5.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(4, 4.5), xytext=(6, 4.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(5, 7.5, 'Stable bistable states', ha='center', fontsize=10)
ax.text(5, 1.5, '- Static storage\n- Random access\n- Dense', ha='center', fontsize=9)

# SCE Shift Register
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SCE Shift Register', fontsize=12, fontweight='bold')

# Draw shift register stages
for i in range(4):
    rect = FancyBboxPatch((1 + i*2, 4), 1.5, 2, boxstyle="round,pad=0.1",
                          facecolor=COLORS['fifo'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(1.75 + i*2, 5, 'DFF', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    if i < 3:
        ax.annotate('', xy=(2.7 + i*2, 5), xytext=(2.5 + i*2, 5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(5, 7.5, 'Data flows through stages', ha='center', fontsize=10)
ax.text(5, 1.5, '- Simple JJ implementation\n- Sequential access only\n- Constant circulation', ha='center', fontsize=9)

# SCE Vortex Memory
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SCE Vortex Memory', fontsize=12, fontweight='bold')

# Draw superconducting loop with trapped flux
circle = plt.Circle((5, 5), 2, fill=False, color=COLORS['primary'], linewidth=3)
ax.add_patch(circle)
ax.plot(5, 5, 'o', markersize=15, color=COLORS['secondary'])
ax.text(5, 5, 'Φ₀', ha='center', va='center', fontsize=12, fontweight='bold')

ax.text(5, 7.5, 'Trapped flux quantum', ha='center', fontsize=10)
ax.text(5, 1.5, '- True static storage\n~ Denser than shift reg\n- Complex read/write', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey insight: SCE trades random access for deterministic dataflow")

---
<a id="2-shift-registers"></a>
# 2. Shift Register Memory
---

The simplest form of SCE memory is the **shift register** - a chain of D flip-flops (DFFs) through which data propagates on each clock cycle.

## RSFQ Shift Register

Each stage consists of:
- A Josephson transmission line (JTL) for signal propagation
- A DFF cell for storage
- Clock distribution to each stage

```
Data In → [DFF] → [DFF] → [DFF] → [DFF] → Data Out
            ↑        ↑        ↑        ↑
           CLK      CLK      CLK      CLK
```

## Characteristics

| Parameter | RSFQ Shift Register |
|-----------|--------------------|
| Clock speed | 20-50 GHz typical |
| Latency | N cycles for N stages |
| Access pattern | Sequential (FIFO) |
| Energy/bit/cycle | ~1 aJ (at 4K) |
| Area/bit | ~100 μm² |

## Use Cases

1. **Pipeline registers**: Balancing paths in pipelined logic
2. **Delay lines**: Matching timing between functional units
3. **Small FIFOs**: Buffering between processing stages

### Limitation: No Random Access

To read bit N, you must shift N times - making random access O(N) instead of O(1).

In [ ]:
# Visualize: Shift register operation over time
fig, ax = plt.subplots(figsize=(12, 5))

n_stages = 6
n_cycles = 8

# Initial data pattern: [1, 0, 1, 1, 0, 1, 0, 0]
data = [1, 0, 1, 1, 0, 1, 0, 0]

for cycle in range(n_cycles):
    for stage in range(n_stages):
        # Which bit is in this stage at this cycle?
        bit_idx = cycle - stage
        if 0 <= bit_idx < len(data):
            color = COLORS['success'] if data[bit_idx] else COLORS['light']
            value = str(data[bit_idx])
        else:
            color = '#e0e0e0'
            value = ''
        
        rect = FancyBboxPatch((stage * 1.5 + 0.5, n_cycles - cycle - 1), 1, 0.8,
                              boxstyle="round,pad=0.05", facecolor=color,
                              edgecolor='black', linewidth=1)
        ax.add_patch(rect)
        ax.text(stage * 1.5 + 1, n_cycles - cycle - 0.6, value,
                ha='center', va='center', fontsize=10, fontweight='bold')

# Labels
for stage in range(n_stages):
    ax.text(stage * 1.5 + 1, n_cycles + 0.3, f'Stage {stage}', ha='center', fontsize=9)

for cycle in range(n_cycles):
    ax.text(0.2, n_cycles - cycle - 0.6, f't={cycle}', ha='right', fontsize=9)

ax.set_xlim(0, n_stages * 1.5 + 1)
ax.set_ylim(-0.5, n_cycles + 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Shift Register: Data [1,0,1,1,0,1,0,0] propagating through 6 stages', 
             fontsize=12, fontweight='bold')

# Arrow showing data flow direction
ax.annotate('', xy=(n_stages * 1.5 + 0.5, n_cycles/2), xytext=(0.5, n_cycles/2),
            arrowprops=dict(arrowstyle='->', color=COLORS['primary'], lw=3))
ax.text(n_stages * 0.75 + 0.5, n_cycles/2 + 0.4, 'Data Flow', ha='center', 
        fontsize=10, color=COLORS['primary'])

plt.tight_layout()
plt.show()

---
<a id="3-vortex-memory"></a>
# 3. Vortex Transitional Memory
---

**Vortex Transitional Memory (VTM)** provides denser storage by trapping flux quanta in superconducting loops.

## Operating Principle

1. **Write**: An SFQ pulse injects a flux quantum (Φ₀) into a storage loop
2. **Store**: The persistent current maintains the trapped flux
3. **Read**: A read pulse extracts or senses the stored flux

## Cell Structure

```
Write Line ──┬── [JJ] ──┬── Read Line
             │          │
             └── L ─────┘
                 ↑
           Storage Loop
```

- **L**: Storage inductor (determines Φ₀ capacity)
- **JJ**: Josephson junctions for write/read switching

## Comparison with Shift Registers

| Property | Shift Register | Vortex Memory |
|----------|---------------|---------------|
| Storage type | Dynamic | Static |
| Density | ~100 μm²/bit | ~50 μm²/bit |
| Access | Sequential | Row-addressable |
| Read | Non-destructive | Destructive (must restore) |
| Complexity | Low | Higher |

## Challenges

- **Destructive reads**: Reading often requires restoring the bit
- **Sense margins**: Distinguishing 0 vs 1 requires careful design
- **Write energy**: Injecting flux requires ~10× read energy

In [ ]:
# Visualize: Vortex memory cell operation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

def draw_vtm_cell(ax, state, title):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # Storage loop
    loop = plt.Circle((5, 5), 2.5, fill=False, color=COLORS['primary'], linewidth=3)
    ax.add_patch(loop)
    
    # Write JJ (left)
    ax.plot([2, 2.5], [5, 5], 'k-', linewidth=2)
    ax.plot([2.5, 2.5], [4.5, 5.5], 'k-', linewidth=3)
    ax.text(1.5, 5, 'W', ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Read JJ (right)
    ax.plot([7.5, 8], [5, 5], 'k-', linewidth=2)
    ax.plot([7.5, 7.5], [4.5, 5.5], 'k-', linewidth=3)
    ax.text(8.5, 5, 'R', ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Inductor (bottom)
    ax.plot([5, 5], [2.5, 1.5], 'k-', linewidth=2)
    for i in range(4):
        ax.add_patch(plt.Circle((5, 1.5 - i*0.3), 0.15, fill=False, color='black', linewidth=2))
    ax.text(5, 0.2, 'L', ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Flux quantum indicator
    if state == 'stored':
        ax.plot(5, 5, 'o', markersize=25, color=COLORS['secondary'], alpha=0.7)
        ax.text(5, 5, 'Φ₀', ha='center', va='center', fontsize=14, fontweight='bold')
        # Circulating current arrow
        arrow = FancyArrowPatch((6.5, 6.5), (3.5, 6.5), 
                                connectionstyle="arc3,rad=0.5",
                                arrowstyle='->', color=COLORS['success'], lw=2)
        ax.add_patch(arrow)
        ax.text(5, 8, 'I_circ', ha='center', fontsize=9, color=COLORS['success'])
    elif state == 'writing':
        ax.annotate('', xy=(3.5, 5), xytext=(1, 5),
                    arrowprops=dict(arrowstyle='->', color=COLORS['danger'], lw=3))
        ax.text(2, 6, 'SFQ\npulse', ha='center', fontsize=9, color=COLORS['danger'])

draw_vtm_cell(axes[0], 'empty', 'Empty Cell (bit = 0)')
draw_vtm_cell(axes[1], 'writing', 'Write Operation')
draw_vtm_cell(axes[2], 'stored', 'Stored Flux (bit = 1)')

plt.tight_layout()
plt.show()

---
<a id="4-sfq-storage"></a>
# 4. SFQ-Based Storage Cells
---

Beyond shift registers and vortex memory, several SFQ-compatible storage cells have been developed:

## 4.1 NDRO Cell (Non-Destructive Read-Out)

Uses a **SQUID** (Superconducting QUantum Interference Device) to sense stored flux without destroying it.

**Advantages:**
- Non-destructive reads (no restore cycle needed)
- Compatible with RSFQ/ERSFQ logic

**Disadvantages:**
- Larger cell area (~200 μm²)
- More complex fabrication

## 4.2 nTron Memory

Uses **nanocryotron (nTron)** devices - superconducting nanowires that switch between superconducting and resistive states.

**Advantages:**
- Very compact
- Low switching energy

**Disadvantages:**
- Slower switching (~ns vs ps for JJ)
- Different fabrication process

## 4.3 Hybrid CMOS-SCE

For large storage requirements, **hybrid architectures** use:
- SCE logic at 4K for computation
- CMOS memory at 77K or room temperature
- High-bandwidth interconnect between temperature stages

This is the practical approach for AI accelerators requiring GB-scale storage.

---
<a id="5-register-fifos"></a>
# 5. Register Files & FIFOs
---

For practical SCE systems, we often need:

1. **Register files**: Small, fast storage with limited random access
2. **FIFOs**: First-in-first-out buffers for streaming data

## FIFO Architecture

FIFOs are natural for SCE because they match the sequential access pattern:

```
Write Port → [Stage 0] → [Stage 1] → ... → [Stage N-1] → Read Port
```

**Key parameters:**
- **Depth**: Number of entries (stages)
- **Width**: Bits per entry
- **Throughput**: One entry per clock cycle

## Register File with Limited Access

For cases requiring some random access, a **banked register file** provides a middle ground:

```
          ┌─────────┐
Write → │ Bank 0  │ ← Read (selected bank)
          ├─────────┤
          │ Bank 1  │
          ├─────────┤
          │ Bank 2  │
          ├─────────┤
          │ Bank 3  │
          └─────────┘
              ↑
         Bank Select
```

- Access any bank in O(1), but within a bank it's sequential
- Trade-off: More banks = more random access, but higher area/complexity

In [ ]:
# Visualize: FIFO vs Banked Register File
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FIFO
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('FIFO Buffer (Depth=8)', fontsize=12, fontweight='bold')

for i in range(8):
    color = COLORS['fifo'] if i < 5 else COLORS['light']
    rect = FancyBboxPatch((i*1.3 + 0.5, 2), 1, 2, boxstyle="round,pad=0.05",
                          facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    if i < 5:
        ax.text(i*1.3 + 1, 3, f'D{4-i}', ha='center', va='center', 
                fontsize=10, color='white', fontweight='bold')

ax.annotate('Write', xy=(0.5, 3), xytext=(-0.5, 3),
            arrowprops=dict(arrowstyle='->', color=COLORS['primary'], lw=2),
            fontsize=10, va='center')
ax.annotate('Read', xy=(11.5, 3), xytext=(10.5, 3),
            arrowprops=dict(arrowstyle='->', color=COLORS['primary'], lw=2),
            fontsize=10, va='center', ha='right')

ax.text(6, 0.5, 'Sequential access only\nPerfect for streaming workloads', 
        ha='center', fontsize=10)

# Banked Register File
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Banked Register File (4 banks × 4 entries)', fontsize=12, fontweight='bold')

for bank in range(4):
    for entry in range(4):
        color = COLORS['register'] if bank == 1 else COLORS['light']
        rect = FancyBboxPatch((entry*1.3 + 3, 5 - bank*1.2), 1, 0.9,
                              boxstyle="round,pad=0.02", facecolor=color,
                              edgecolor='black', linewidth=1)
        ax.add_patch(rect)
    ax.text(2.5, 5.4 - bank*1.2, f'Bank {bank}', ha='right', va='center', fontsize=9)

# Bank select arrow
ax.annotate('', xy=(8.5, 3.8), xytext=(10, 3),
            arrowprops=dict(arrowstyle='->', color=COLORS['danger'], lw=2))
ax.text(10.5, 3, 'Select', fontsize=9, color=COLORS['danger'])

ax.text(6, 0.5, 'O(1) bank access + sequential within bank\nGood for tiled computation', 
        ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
<a id="6-external-memory"></a>
# 6. External Memory Integration
---

For AI workloads requiring gigabytes of storage (model weights, KV cache), on-chip SCE memory is insufficient. The solution: **external memory at higher temperature stages**.

## Memory Hierarchy

```
┌─────────────────────────────────────────┐
│            4K Stage (SCE)               │
│  ┌─────────────────────────────────┐    │
│  │  Compute Array (Systolic/AQFP)  │    │
│  │      + Local Registers          │    │
│  │      + FIFOs / Accumulators     │    │
│  └─────────────────────────────────┘    │
│              ↕ High-BW Link             │
└─────────────────────────────────────────┘
              ↕ Cryo Interface
┌─────────────────────────────────────────┐
│         77K or 300K Stage               │
│  ┌─────────────────────────────────┐    │
│  │        HBM / DDR Memory         │    │
│  │         (GB scale)              │    │
│  └─────────────────────────────────┘    │
└─────────────────────────────────────────┘
```

## Interface Challenges

| Challenge | Solution |
|-----------|----------|
| Thermal load from I/O | Minimize wire count, use SerDes |
| Signal integrity | Impedance matching across temp stages |
| Bandwidth | Wide parallel buses or high-speed serial |
| Latency | Prefetching, double buffering |

## HBM Bandwidth

Modern HBM (High Bandwidth Memory) provides:
- **HBM2e**: ~460 GB/s per stack
- **HBM3**: ~800 GB/s per stack
- **HBM3e**: ~1.2 TB/s per stack

With multiple stacks, total bandwidth can reach several TB/s - matching or exceeding GPU memory bandwidth.

In [ ]:
# Visualize: Memory bandwidth requirements vs HBM capability
fig, ax = plt.subplots(figsize=(10, 6))

# Workload bandwidth requirements (GB/s)
workloads = {
    'LLM Inference\n(batch=1)': 140,
    'LLM Inference\n(batch=32)': 500,
    'LLM Inference\n(batch=256)': 1200,
    'Training\n(Large batch)': 2000,
}

# Memory capabilities
memory_bw = {
    'HBM2e (1 stack)': 460,
    'HBM3 (1 stack)': 800,
    'HBM3e (2 stacks)': 2400,
    'HBM3e (4 stacks)': 4800,
}

x = np.arange(len(workloads))
width = 0.35

bars1 = ax.bar(x - width/2, list(workloads.values()), width, 
               label='Workload Requirement', color=COLORS['primary'])

# Add HBM capability lines
for name, bw in memory_bw.items():
    ax.axhline(y=bw, color=COLORS['secondary'], linestyle='--', alpha=0.7)
    ax.text(len(workloads)-0.5, bw + 50, name, fontsize=9, 
            color=COLORS['secondary'], ha='right')

ax.set_ylabel('Bandwidth (GB/s)', fontsize=11)
ax.set_title('Memory Bandwidth: Workload Requirements vs HBM Capability', 
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(list(workloads.keys()))
ax.legend(loc='upper left')
ax.set_ylim(0, 5500)

plt.tight_layout()
plt.show()

print("Key insight: HBM provides sufficient bandwidth for most inference workloads.")
print("The challenge is the cryo interface, not raw memory bandwidth.")

---
<a id="7-memory-ai"></a>
# 7. Memory for AI Workloads
---

The memory architecture for an SCE AI accelerator is fundamentally different from a GPU.

## GPU Memory Architecture

GPUs use large on-chip SRAM for:
- **L1/L2 caches**: Hiding memory latency
- **Shared memory**: Inter-thread communication
- **Register files**: Per-thread state

This enables **random access patterns** and **data reuse** across threads.

## SCE-Optimized Architecture

For SCE, we optimize for **deterministic dataflow**:

| Component | GPU Approach | SCE Approach |
|-----------|-------------|---------------|
| Weight storage | L2 cache + HBM | HBM (streamed) |
| Activation buffer | Shared memory | FIFOs between stages |
| Partial sums | Register file | Edge accumulators |
| KV cache | HBM | HBM (sequential access) |

## Why This Works for LLM Inference

LLM inference has **highly predictable** memory access patterns:

1. **Weights**: Streamed once per token, same order every time
2. **KV cache**: Append-only (new tokens) + sequential read (attention)
3. **Activations**: Flow through layers in fixed order

No random access needed → **FIFOs + accumulators can replace SRAM entirely**.

### Accumulator Sizing

For a systolic array processing matrix tiles:

```
Accumulator size = Batch × Tile_N × Bytes_per_element
```

For LLM inference with batch=32 and tile_N=256 (FP16):
```
Accumulator size = 32 × 256 × 2 bytes = 16 KB
```

This is **tiny** compared to GPU SRAM (tens of MB), and can be implemented with SCE registers.

In [ ]:
# Visualize: GPU vs SCE memory architecture comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# GPU Architecture
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('GPU Memory Architecture', fontsize=12, fontweight='bold')

# HBM
rect = FancyBboxPatch((0.5, 0.5), 9, 1.5, boxstyle="round,pad=0.1",
                       facecolor=COLORS['primary'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 1.25, 'HBM (80 GB)', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# L2 Cache
rect = FancyBboxPatch((1, 2.5), 8, 1.2, boxstyle="round,pad=0.1",
                       facecolor=COLORS['sram'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 3.1, 'L2 Cache (50 MB)', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Shared Memory / L1
for i in range(4):
    rect = FancyBboxPatch((1.5 + i*2, 4.2), 1.5, 1, boxstyle="round,pad=0.1",
                          facecolor=COLORS['register'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
ax.text(5, 4.7, 'Shared Mem + L1 (per SM)', ha='center', va='center', fontsize=9, fontweight='bold')

# Register File
for i in range(4):
    rect = FancyBboxPatch((1.5 + i*2, 5.7), 1.5, 0.8, boxstyle="round,pad=0.1",
                          facecolor=COLORS['success'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
ax.text(5, 6.1, 'Register File (256 KB/SM)', ha='center', va='center', fontsize=9, fontweight='bold')

# Compute
rect = FancyBboxPatch((2, 7), 6, 1.5, boxstyle="round,pad=0.1",
                       facecolor=COLORS['dark'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 7.75, 'Tensor Cores', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.text(5, 9, 'Complex hierarchy\nRandom access patterns', ha='center', fontsize=10)

# SCE Architecture
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SCE Memory Architecture', fontsize=12, fontweight='bold')

# HBM (external)
rect = FancyBboxPatch((0.5, 0.5), 9, 1.5, boxstyle="round,pad=0.1",
                       facecolor=COLORS['primary'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 1.25, 'HBM @ 77K/300K', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Cryo interface
ax.plot([5, 5], [2, 2.8], 'k--', linewidth=2)
ax.text(5.5, 2.4, 'Cryo\nInterface', ha='left', fontsize=9)

# Input FIFO
rect = FancyBboxPatch((1, 3), 3, 1, boxstyle="round,pad=0.1",
                       facecolor=COLORS['fifo'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(2.5, 3.5, 'Input FIFO', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Weight FIFO
rect = FancyBboxPatch((6, 3), 3, 1, boxstyle="round,pad=0.1",
                       facecolor=COLORS['fifo'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(7.5, 3.5, 'Weight FIFO', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Compute array
rect = FancyBboxPatch((2, 4.5), 6, 2, boxstyle="round,pad=0.1",
                       facecolor=COLORS['dark'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 5.5, 'AQFP Systolic Array', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Edge accumulators
rect = FancyBboxPatch((2, 7), 6, 0.8, boxstyle="round,pad=0.1",
                       facecolor=COLORS['register'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 7.4, 'Edge Accumulators (32 KB)', ha='center', va='center', fontsize=10, fontweight='bold')

# Output FIFO
rect = FancyBboxPatch((3.5, 8.2), 3, 0.8, boxstyle="round,pad=0.1",
                       facecolor=COLORS['fifo'], edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(5, 8.6, 'Output FIFO', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.text(5, 9.5, 'Simple hierarchy\nDeterministic dataflow', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nKey insight: SCE replaces complex cache hierarchy with simple FIFOs + accumulators.")
print("This works because LLM inference has predictable, streaming access patterns.")

---
<a id="summary"></a>
# Summary
---

## Key Takeaways

1. **CMOS SRAM doesn't translate to SCE**: No transistor bistability means different storage primitives

2. **SCE storage options**:
   - Shift registers: Simple, sequential access
   - Vortex memory: Denser, static storage
   - FIFOs: Natural fit for streaming workloads
   - External HBM: For GB-scale storage

3. **Memory architecture for AI**:
   - LLM inference has predictable access patterns
   - FIFOs + edge accumulators can replace SRAM
   - Accumulator requirements are small (~32 KB for typical configs)

4. **The key insight**: Match memory architecture to workload patterns, not GPU conventions

## Looking Ahead

In the next lecture, we'll see how to compose these memory elements with logic gates into complete functional units using **Circuit Design & Integration** techniques.